In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

## Graphical Representation

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import * as tslab from "tslab";

In [ ]:
interface HeapNode {
    mIndex: number;
    [key: string]: any;
}

type HeapElement = [number, HeapNode];

async function heapToDot(A: HeapElement[]): Promise<void> {
    const n = A.length;
    let dot = 'digraph {\n';
    dot += 'node [shape=record];\n';
    for (let k = 0; k < n; k++) {
        const [p, o] = A[k];
        if (String(p) !== String(o)) {
            dot += `${k} [label="{${p}|${o}|${o.mIndex}}", style=rounded];\n`;
        } else {
            dot += `${k} [label="{${p}|${k}}", style=rounded];\n`;
        }
    }
    dot += '\n';
    for (let k = 0; k < Math.floor(n / 2); k++) {
        if (2 * k + 1 < n) {
            dot += `${k} -> ${2 * k + 1};\n`;
        }
        if (2 * k + 2 < n) {
            dot += `${k} -> ${2 * k + 2};\n`;
        }
    }
    dot += '}\n';
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    tslab.display.html(svg);
}

The function `toDot` takes four arguments:
- `A` is an array of natural numbers of length $n$,
- `f` is a natural number such that $0 \leq f < n$ holds,
- `g` is a natural number such that $f < g < n$ holds,
- `u` is a natural number such that $0 \leq u < n$ holds.
  This argument is optional.

The function returns a graphical representation of the array `A` as a heap. 
This graphical representation is stored as a directed graph with an encoding suitable for `graphviz`. 

The part `A[0:g]` is represented as a binary tree, while the part `A[g:]` is represented
as an array.  Furthermore, all indexes in the range `A[k:g]` satisfy the heap condition.  The nodes in the range `[0:k-1]`
are colored red.  If `u` is set, the node `A[u]` is colored orange.

# Priority Queues implemented as Heaps

## Auxiliary Methods

In [ ]:
class Node implements HeapNode {
    mValue: any;
    mIndex: number = 0;
    constructor(value: any) {
        this.mValue = value;
    }
    toString(): string {
        return this.mValue.toString();
    }
    static compare(a: Node, b: Node): number {
        if (a.mValue < b.mValue) return -1;
        if (a.mValue > b.mValue) return 1;
        return 0;
    }
}

In [ ]:
function less(a: HeapElement, b: HeapElement): boolean {
    const [pa, oa] = a;
    const [pb, ob] = b;
    if (pa < pb) return true;
    if (pa > pb) return false;
    return Node.compare(oa as Node, ob as Node) < 0;
}

The function call `swap(A, i, j)` takes an array `A` and  two indexes `i` and `j` and exchanges the elements at these indexes.

In [ ]:
function swap(A: HeapElement[], i: number, j: number): void {
    const [pi, oi] = A[i];
    const [pj, oj] = A[j];
    oi.mIndex = j;
    oj.mIndex = i;
    A[i] = [pj, oj];
    A[j] = [pi, oi];
}

The function `ascend` takes two arguments:
- `A` is an array
- `k` is an index into the array `A`.

   Therefore we have $k \in \bigl\{0, \cdots, \texttt{len}(A)-1\bigr\}$.

The array `A` represents a *heap*.  However, the <em style="color:blue">heap condition</em> might be violated 
at index `k`: It might be the case that the element at this index is to small and needs to rise to the top
of the heap.  The function `ascend` will fix the heap condition and will rise the element `A[k]` as much 
as is necessary to turn `A` into a heap.

In [ ]:
function ascend(A: HeapElement[], k: number): number {
    while (k > 0) {
        const p = Math.floor((k - 1) / 2);
        if (less(A[k], A[p])) {
            swap(A, p, k);
            k = p;
        } else {
            return k;
        }
    }
    return 0;
}

The function `descend(A)` takes one argument `A` where `A` is an array that is organized as a heap,
but possibly has its heap condition violated at its root, i.e. at index `0`.  The
purpose of the procedure `descend` is to restore the heap condition at the root.
We initialize a variable `k` as `0` and the `while`-loop proceeds as follows: 
- We compute the index `j` of the left subtree below index `k`.
- We check whether there also is a right subtree at position `j+1`.
  
  This is the case if `j + 1 < n` where `n = A.length - 1`.  
- If the heap condition is violated at index `k`, we exchange the element at  position `k` 
  with the child that has the higher priority, i.e. the child that is smaller. 
- Next, we check in line 7 whether the heap condition is violated at index `k`.  
  If the heap condition is satisfied, there is nothing left to do and the procedure returns.  
  
- Otherwise, the element at position `k` is swapped with
  the element at position `j`.  
  
  Of course, after this swap it is possible that the heap condition is
  violated at position `j`.  Therefore,  `k` is set to `j` and the `while`-loop continues
  as long as the node at position `k` has at least one child, i.e. as long as 
  `2 * k + 1 <= n`.

In [ ]:
function descend(A: HeapElement[]): void {
    const n = A.length - 1;
    let k = 0;
    while (2 * k + 1 <= n) {
        let j = 2 * k + 1;
        if (j + 1 <= n && less(A[j + 1], A[j])) {
            j += 1;
        }
        if (less(A[k], A[j])) {
            return;
        }
        swap(A, k, j);
        k = j;
    }
}

## Implementing the API

The function `insert(H, x)` takes two arguments:
- `H` is a heap that is represented as an array.
- `x` is a pair of the form `(p, o)` where
  - `p` is a natural number interpreted as a priority.  The smaller the number, the higher the priority.
  - `o` is an object.  
  
    Every object `o` knows its index in the heap via the member variable `o.mIndex`.
    
This method inserts the pair `x` into the heap `H`.  Furthermore, the object `o` is modified so that it remembers
the index at which it is stored in `H`.  This is done by storing this index in `o.mIndex`.

In [ ]:
function insert(H: HeapElement[], x: HeapElement): void {
    const n = H.length;
    H.push(x);
    const [_, o] = x;
    o.mIndex = n;
    const k = ascend(H, n);
    o.mIndex = k;
}

The function `elevate(H, o, p)` takes three arguments.
- `H` is an array that is organized as a heap.
- `o` is an object that occurs in the heap `H` at index `o.mIndex`, i.e. we have `H[o.mIndex] = p_old, o.mIndex`,
  where `p_old` is the priority that was used when `o` was stored in `H`.
- `p` is the new priority of `o` in `H`.  This priority must be higher than the priority `p_old`, i.e. we must have `p < p_old`.

The function call `elevate(H, o, p)` elevates the priority of the object `o` to `p` in `H` and takes care that `o` is stored further up in `H` 
so that the heap property of `H` is maintained.

In [ ]:
function elevate(H: HeapElement[], o: HeapNode, p: number): void {
    const k = o.mIndex;
    H[k] = [p, o];
    ascend(H, k);
}

In [ ]:
function remove(H: HeapElement[]): HeapElement {
    const [pFirst, oFirst] = H[0];
    const [pLast, oLast] = H[H.length - 1];
    oLast.mIndex = 0;
    H[0] = [pLast, oLast];
    H.pop();
    descend(H);
    return [pFirst, oFirst];
}

## Testing

In [ ]:
async function demo1(): Promise<void> {
    const L: HeapElement[] = [];
    for (let i = 0; i < 26; i++) {
        const c = String.fromCharCode(97 + i);
        L.push([i, new Node(c)]);
    }
    const [_, w] = L[22]; // das ist der Node für 'w'
    const H: HeapElement[] = [];
    for (let i = 0; i < L.length; i++) {
        insert(H, L[i]);
    }
    await heapToDot(H);
    console.log('Elevating "w" to priority 2:');
    elevate(H, w, 2);
    await heapToDot(H);
}

In [ ]:
await demo1();

In [ ]:
async function heapSort(L: HeapElement[]): Promise<number[]> {
    const H: HeapElement[] = [];
    for (const x of L) {
        insert(H, x);
    }
    await heapToDot(H);
    const S: number[] = [];
    while (H.length > 0) {
        const [p, _] = remove(H);
        await heapToDot(H);
        S.push(p);
    }
    return S;
}

In [ ]:
async function demo2(): Promise<void> {
    const L: HeapElement[] = [];
    for (let n = 0; n < 12; n++) {
        const randNum = Math.floor(Math.random() * 199) + 1; // random 1-200
        L.push([randNum, new Node(randNum)]);
    }
    console.log("L = ", L);
    const S = await heapSort(L);
    console.log("S = ", S);
}

In [ ]:
await demo2();